In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
from itertools import batched
from tqdm import tqdm
import json
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import torch
from mtrain.utils import (
    show,
    mkdir,
    overlay_mask_on_img as OV,
    draw_grid_cv2,
    show_single_channel_red_green_black,
    DiskBooleanMask,
    DiskImage,
    stack_batch,
)
from mtrain.interp import cds
from mtrain.interp.analysis import (
    list_layers,
    get_layer_data,
    find_weights_discrepancies,
    to_activation_id,
    to_bias_id,
    to_weight_id,
    get_weights_and_acts,
    get_input_bbox_padded,
    get_input_patch_padded,
    get_per_channel_conv,
    BBox,
    run_on_layer,
    normalize,
    run_and_show_for_input,
    print_conv_stats,
    RunResult,
    print_model_weights_bias_stats,
)

In [ ]:
def N(img_tensor):
    mean_vals = [0.485, 0.456, 0.406]
    std_vals = [0.229, 0.224, 0.225]
    return normalize(img_tensor, mean_vals, std_vals)

def print_green_and_red_stat(per_chan, red_idx, green_idx):
    green_edges = [p[green_idx] for p in per_chan]
    red_edges = [p[red_idx] for p in per_chan]

    print("\ngreen maxes")
    for i, green in enumerate(green_edges):
        print("\t", i, green.max())
    print("\nred mins")
    for i, red in enumerate(red_edges):
        print("\t", i, red.min())

# Analysis

In [ ]:
WEIGHTS_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/interpretation/simplenet/weights"
)
ACTIVATIONS_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/interpretation/simplenet/analysis/trash_for_simple_net"
)

print_model_weights_bias_stats(ACTIVATIONS_DIR, WEIGHTS_DIR)

In [ ]:
from itertools import islice

params = list(islice(get_weights_and_acts(WEIGHTS_DIR, ACTIVATIONS_DIR), 4))
print("extracted params")
for p in params:
    print(
        "\t",
        p["activation"]["metadata"]["layer_type"],
        p["activation"]["metadata"]["layer_name"],
    )

print("batchnorm same as relu?", np.all(params[1]["activation"]["data"] == params[2]["activation"]["data"]))

In [ ]:
from fastai.vision.all import load_learner

learner = load_learner(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/interp/standard_resnet18_neg_mask/iter-8.pkl"
)

In [ ]:
def get_orig_preds():
    with open("/Users/hariomnarang/Desktop/personal/roads/datasets/interpretation/smallnet/preds.json") as f:
        return json.load(f)

def run_pred_test_set(learner, bs=16, samples=-1):
    learner.model.eval()

    path_by_orig = get_orig_preds()
    path_by_pred = {}
    paths = sorted((k for k in path_by_orig))
    if samples > -1:
        paths = paths[:samples]

    for batch in tqdm(list(batched(paths, bs))):
        dl = learner.dls.test_dl(batch)
        _, _, classes = learner.get_preds(dl=dl, with_decoded=True)
        assert len(batch) == len(classes)
        for p, c in zip(batch, classes):
            if c == 0:
                clazz = "other"
            elif c == 1:
                clazz = "trash"
            else:
                raise Exception(f"invalid class {c}")
            path_by_pred[p] = clazz

    return path_by_pred, path_by_orig


def get_changed_preds(p_by_pred, p_by_orig):
    changes = []
    for p in p_by_pred:
        if p not in p_by_orig:
            continue
        pred = p_by_pred[p]
        orig = p_by_orig[p]
        if pred != orig:
            changes.append({"path": p, "pred": pred, "orig": orig})

    return changes


In [ ]:
# pbypred = run_batch_pred(learner)

In [ ]:
layer_id = params[0]["activation"]["metadata"]["layer_name"]
layer_id

# 1 channel white, all others black

In [ ]:
white, black = cds.white(50), cds.black(50)


white0 = N(stack_batch([white, black, black]))
white1 = N(stack_batch([black, white, black]))
white2 = N(stack_batch([black, black, white]))

In [ ]:
_ = run_and_show_for_input(
    layer_id, white0, learner.model, 0, figsize=(10, 10), viztype="global"
)

# Strong alignment

In [ ]:
step_edge = cds.step_edge(50, 0, 20, 0, 1)


step_edge1 = N(torch.stack([step_edge, step_edge, step_edge]).unsqueeze(0))

res = run_and_show_for_input(
    layer_id, step_edge1, learner.model, 0, figsize=(10, 10), viztype="gray"
)
print_conv_stats(res)

In [ ]:
res["kernel"][0]

In [ ]:
ip = stack_batch(res["input"])
patch9, bbox9 = get_input_patch_padded(ip, layer_id, learner.model, (9, 10))
patch9 = patch9.numpy()[0]
patch10, bbox10 = get_input_patch_padded(ip, layer_id, learner.model, (10, 10))
patch10 = patch10.numpy()[0]

print("patch shape", patch9.shape, patch10.shape)
print_green_and_red_stat(res["per_chan"], 9, 10)

## Kernel 0

In [ ]:
idx = 0
show_single_channel_red_green_black(
    [
        res["input"][idx],
        res["per_chan"][idx],
        res["kernel"][idx],
        patch9[idx],
        res["kernel"][idx],
        patch9[idx] * res["kernel"][idx],
        patch10[idx],
        res["kernel"][idx],
        patch10[idx] * res["kernel"][idx],
    ],
    ncols=3,
    viztype="local",
)

## Kernel 1

In [ ]:
idx = 1
show_single_channel_red_green_black(
    [
        res["input"][idx],
        res["per_chan"][idx],
        res["kernel"][idx],
        patch9[idx],
        res["kernel"][idx],
        patch9[idx] * res["kernel"][idx],
        patch10[idx],
        res["kernel"][idx],
        patch10[idx] * res["kernel"][idx],
    ],
    ncols=3,
    viztype="local",
)

## Kernel 2

In [ ]:
idx = 2
show_single_channel_red_green_black(
    [
        res["input"][idx],
        res["per_chan"][idx],
        res["kernel"][idx],
        patch9[idx],
        res["kernel"][idx],
        patch9[idx] * res["kernel"][idx],
        patch10[idx],
        res["kernel"][idx],
        patch10[idx] * res["kernel"][idx],
    ],
    ncols=3,
    viztype="local",
)

# Weak alignment

In [ ]:
se = cds.step_edge(50, 0, 21)
step_edge2 = N(stack_batch([se, se, se]))
res = run_and_show_for_input(
    layer_id, step_edge2, learner.model, 0, figsize=(10, 10), viztype="local"
)
print_conv_stats(res)

In [ ]:
ip = stack_batch(res["input"])
patch9, bbox9 = get_input_patch_padded(ip, layer_id, learner.model, (9, 10))
patch9 = patch9.numpy()[0]
patch10, bbox10 = get_input_patch_padded(ip, layer_id, learner.model, (11, 10))
patch10 = patch10.numpy()[0]

# green_edges =[p[9] for p in res["per_chan"]]
# red_edges =[p[11] for p in res["per_chan"]]

# for i, (green, red) in enumerate(zip(green_edges, red_edges)):
#     print(i, "green max", green.numpy().max(), "red min", red.numpy().min())

print("patch shape", patch9.shape, patch10.shape)
print_green_and_red_stat(res["per_chan"], 11, 9)

## Kernel 0

In [ ]:
idx = 0
show_single_channel_red_green_black(
    [
        res["input"][idx],
        res["per_chan"][idx],
        res["kernel"][idx],
        patch9[idx],
        res["kernel"][idx],
        patch9[idx] * res["kernel"][idx],
        patch10[idx],
        res["kernel"][idx],
        patch10[idx] * res["kernel"][idx],
    ],
    ncols=3,
    viztype="local",
)

## Kernel 1

In [ ]:
idx = 1
show_single_channel_red_green_black(
    [
        res["input"][idx],
        res["per_chan"][idx],
        res["kernel"][idx],
        patch9[idx],
        res["kernel"][idx],
        patch9[idx] * res["kernel"][idx],
        patch10[idx],
        res["kernel"][idx],
        patch10[idx] * res["kernel"][idx],
    ],
    ncols=3,
    viztype="local",
)

## Kernel 2

In [ ]:
idx = 2
show_single_channel_red_green_black(
    [
        res["input"][idx],
        res["per_chan"][idx],
        res["kernel"][idx],
        patch9[idx],
        res["kernel"][idx],
        patch9[idx] * res["kernel"][idx],
        patch10[idx],
        res["kernel"][idx],
        patch10[idx] * res["kernel"][idx],
    ],
    ncols=3,
    viztype="local",
)

# Tweak and run

I'll now try changing the kernel and running the model.  

Kernel 0 and 1 are aligned and give strong activations in strong alignment. lets flip the sign of kernel 1 so that they compete against one another.  

## Load learners

In [ ]:
from fastai.vision.all import load_learner
from mtrain.interp.cds.test_set import get_path_by_orig_predictions
from mtrain.utils import show_single_channel_red_green_black
import torch
import matplotlib.pyplot as plt


def _load_learner():
    learner = load_learner(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/models/interp/standard_resnet18_neg_mask/iter-8.pkl"
    )
    _ = learner.model.eval()
    return learner

learner = _load_learner()
original_learner = _load_learner()

p_by_pred = get_path_by_orig_predictions()
def run_learner_and_find_changes(learner, p_by_pred):
    changes = []
    for p, label in p_by_pred.items():
        pred = learner.predict(p)[0]
        if pred != label:
            changes.append((p, label, pred))
    return changes

In [ ]:
layer_id = "0.0"
layer = learner.model.get_submodule(layer_id)
kernel0 = layer.weight[0]
k0 = kernel0.detach().clone()
k0[1] = -k0[1]
show_single_channel_red_green_black(
    [k for k in kernel0.detach().numpy()] + [k for k in k0]
, ncols=3)

In [ ]:
with torch.no_grad():
    layer.weight[0] = k0

# confirm
layer = learner.model.get_submodule(layer_id)
kernel0 = layer.weight[0].detach()
print(kernel0.shape)
show_single_channel_red_green_black([k for k in kernel0], ncols=3)

In [ ]:
p_by_pred, p_by_orig = run_pred_test_set(learner)

# changes = run_learner_and_find_changes(learner, p_by_pred)
# changes

In [ ]:
changes = get_changed_preds(p_by_pred, p_by_orig)

In [ ]:
plt.imshow(plt.imread(changes[0]["path"]))

In [ ]:
from mtrain.interp.analysis import run_and_save_for_input
from mtrain.utils import save_single_channel_red_green_black

def test_for_path_with_report(p, layer_id, original_learner, learner, dest_dir):
      from pathlib import Path
      import shutil

      dest_path = Path(dest_dir)
      dest_path.mkdir(exist_ok=True, parents=True)

      # Create unique prefix for this path
      path_name = Path(p).stem

      # Copy original image to dest
      original_img_path = dest_path / f"{path_name}_original.png"
      shutil.copy2(p, original_img_path)

      # Process the image
      t0 = original_learner.dls.test_dl([p]).one_batch()[0]

      # Save original kernel behavior
      original_analysis_path = dest_path / f"{path_name}_original_analysis.png"
      res_original = run_and_save_for_input(
          layer_id, t0, original_learner.model, 0,
          original_analysis_path, viztype="local"
      )

      # Save new kernel behavior
      new_analysis_path = dest_path / f"{path_name}_new_analysis.png"
      res_test = run_and_save_for_input(
          layer_id, t0, learner.model, 0,
          new_analysis_path, viztype="local"
      )

      # Save per channel comparison
      comparison_path = dest_path / f"{path_name}_comparison.png"
      save_single_channel_red_green_black(
          [
              res_original["per_chan"][0],
              res_original["per_chan"][1],
              res_test["per_chan"][0],
              res_test["per_chan"][1],
              res_original["output"],
              res_test["output"],
          ],
          comparison_path,
          figsize=(15, 15),
          viztype="local",
      )

      # Create markdown report for this image
      md_content = f"""# Analysis Report: {path_name}

  **Layer:** `{layer_id}`

  ## Original Image
  ![Original]({path_name}_original.png)

  ## Original Kernel Analysis
  ![Original Analysis]({path_name}_original_analysis.png)

  ## New Kernel Analysis
  ![New Analysis]({path_name}_new_analysis.png)

  ## Per Channel Comparison
  ![Comparison]({path_name}_comparison.png)
  """

      # Write markdown report
      report_path = dest_path / f"{path_name}_report.md"
      with open(report_path, 'w') as f:
          f.write(md_content)

      return report_path, res_original, res_test

In [ ]:
import shutil
from mtrain.interp.analysis import test_for_path_all_viztypes
DEST = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/interpretation/smallnet/reports/layer_0_kernel_0")

learner.model.eval()
original_learner.model.eval()
total = len(changes)
for i, c in tqdm(enumerate(changes), total=total):
    with torch.no_grad():
        dest_dir = DEST / str(i)
        # shutil.rmtree(dest_dir, True)
        if dest_dir.exists():
            continue
        mkdir(dest_dir)
        test_for_path_all_viztypes(
            c["path"], layer_id, original_learner, learner, dest_dir,
        )

In [ ]:
def test_for_path(p, layer_id, original_learner, learner):
    t0 = original_learner.dls.test_dl([p]).one_batch()[0]
    print("original kernel behavior")
    res_original = run_and_show_for_input(layer_id, t0, original_learner.model, 0, viztype="local")
    print("new kernel behavior")
    res_test = run_and_show_for_input(layer_id, t0, learner.model, 0, viztype="local")
    print("per channel comparisons")
    show_single_channel_red_green_black(
        [
            res_original["per_chan"][0],
            res_original["per_chan"][1],
            res_test["per_chan"][0],
            res_test["per_chan"][1],
            res_original["output"],
            res_test["output"],
        ],
        (15, 15),
        viztype="local",
    )

def show_patch_for_path(p, layer_id, original_learner, bbox):
    t0 = original_learner.dls.test_dl([p]).one_batch()[0]
    orig_ip_patch, orig_ip_box = get_input_bbox_padded(
        t0, layer_id, original_learner.model, bbox,
    )

    show_single_channel_red_green_black(
        [
            orig_ip_patch[0][0],
            orig_ip_patch[0][1],
        ]
    )

test_for_path(changes[0]["path"], layer_id, original_learner, learner)
# show_patch_for_path(changes[0]["path"])

We already see a change when we flipped the sign of the middle kernel. The kernel now is destructive to its sibling.  
Need to check the activations now.  

In [ ]:
changed_path = changes[0][0]
plt.imshow(plt.imread(changed_path))

### check original model on input

In [ ]:
# img = plt.imread(changed_path)
dl = original_learner.dls.test_dl([changed_path])
dlt = learner.dls.test_dl([changed_path])

# check both tensors for our different learners are same
t0 = dl.one_batch()[0]
t1 = dlt.one_batch()[0]

torch.allclose(t0, t1), t0.shape, t1.shape

In [ ]:
res_original = run_and_show_for_input(layer_id, t0, original_learner.model, 0, viztype="global")

In [ ]:
res_test = run_and_show_for_input(layer_id, t0, learner.model, 0, viztype="global")

In [ ]:
show_single_channel_red_green_black(
    [
        res_original["per_chan"][0],
        res_original["per_chan"][1],
        res_test["per_chan"][0],
        res_test["per_chan"][1],
        res_original["output"],
        res_test["output"],
    ],
    (15, 15),
    viztype="local",
)

In [ ]:
orig_ip_patch, orig_ip_box = get_input_bbox_padded(
    t0, layer_id, original_learner.model, BBox(18, 5, 21, 12)
)
print(orig_ip_patch.shape)
show_single_channel_red_green_black(
    [
        orig_ip_patch[0][0],
        orig_ip_patch[0][1],
    ]
)

The patch is the strongest horizontal edge in the input which is > 7 pixels vertically in length.  
In this experiment, it seems the model is depending a lot on this specific part of the slipper. Let's look at the closeup to see if it looks anything like a garbage piece.  

In [ ]:
img = plt.imread(changed_path)
cbox = orig_ip_box

plt.imshow(img[cbox.y0 : cbox.y1, cbox.x0 : cbox.x1])

## zero out the last kernel

I already tried inverting it, its not giving any changes, lets try with zeroed out kernel

In [ ]:
learner = _load_learner()
original_learner = _load_learner()

In [ ]:
layer_id = "0.0"
layer = learner.model.get_submodule(layer_id)
kernel0 = layer.weight[0]
k0 = kernel0.detach().clone()
k0[2] = torch.zeros(k0[2].shape)
show_single_channel_red_green_black(
    [k for k in kernel0.detach().numpy()] + [k for k in k0]
, ncols=3)

In [ ]:
with torch.no_grad():
    layer.weight[0] = k0

# confirm
layer = learner.model.get_submodule(layer_id)
kernel0 = layer.weight[0].detach()
print(kernel0.shape)
show_single_channel_red_green_black([k for k in kernel0], ncols=3)

In [ ]:
changes = run_learner_and_find_changes(learner, p_by_pred)
changes

I tried both inverting and zeroing out the kernel, no changes found in any of them, this layer of the kernel does not seem to have a lot of direct effect

# Vertical edges

The kernel does see vertical edges, although its very mild

In [ ]:
learner = _load_learner()
black = cds.black(50)
step = cds.step_edge(50, 90, 25)
layer_id = "0.0"

ip_batch = N(stack_batch([step, step, step]))
ip_batch.shape

In [ ]:
res = run_and_show_for_input(layer_id, ip_batch, learner.model, 0, viztype="global")
print_conv_stats(res)

In [ ]:
plt.imshow(res["per_chan"][2][:, 10:15], cmap="gray")

In [ ]:
ip = stack_batch(res["input"])
patch11, bbox11 = get_input_patch_padded(ip, layer_id, learner.model, (10, 11))
patch11 = patch11.numpy()[0]
patch12, bbox12 = get_input_patch_padded(ip, layer_id, learner.model, (10, 12))
patch12 = patch12.numpy()[0]
patch13, bbox13 = get_input_patch_padded(ip, layer_id, learner.model, (10, 13))
patch13 = patch13.numpy()[0]

print("patch shape", patch11.shape, patch12.shape, patch13.shape)
print("10,11")
print_green_and_red_stat(res["per_chan"], 10, 11)
print("10,12")
print_green_and_red_stat(res["per_chan"], 10, 11)
print("10,13")
print_green_and_red_stat(res["per_chan"], 10, 11)

In [ ]:
print(res["per_chan"][0][4][10])
print(res["per_chan"][0][4][11])
print(res["per_chan"][0][4][12])
print(res["per_chan"][0][4][13])
print(res["per_chan"][0][4][14])
print("---")
print(res["per_chan"][1][4][10])
print(res["per_chan"][1][4][11])
print(res["per_chan"][1][4][12])
print(res["per_chan"][1][4][13])
print(res["per_chan"][1][4][14])
print("---")
print(res["per_chan"][2][4][10])
print(res["per_chan"][2][4][11])
print(res["per_chan"][2][4][12])
print(res["per_chan"][2][4][13])
print(res["per_chan"][2][4][14])

## Kernel 0

In [ ]:
idx = 0
show_single_channel_red_green_black(
    [
        res["input"][idx], res["per_chan"][idx], res["kernel"][idx],
        patch11[idx], res["kernel"][idx], patch11[idx] * res["kernel"][idx],
        patch12[idx], res["kernel"][idx], patch12[idx] * res["kernel"][idx],
        patch13[idx], res["kernel"][idx], patch13[idx] * res["kernel"][idx],
    ],
    ncols=3,
    viztype="gray",
)

## Kernel 1

In [ ]:
## Kernel 0
idx = 1
show_single_channel_red_green_black(
    [
        res["input"][idx], res["per_chan"][idx], res["kernel"][idx],
        patch11[idx], res["kernel"][idx], patch11[idx] * res["kernel"][idx],
        patch12[idx], res["kernel"][idx], patch12[idx] * res["kernel"][idx],
        patch13[idx], res["kernel"][idx], patch13[idx] * res["kernel"][idx],
    ],
    ncols=3,
    viztype="local",
)

## Kernel 2

In [ ]:
## Kernel 0
idx = 2
show_single_channel_red_green_black(
    [
        res["input"][idx], res["per_chan"][idx], res["kernel"][idx],
        patch11[idx], res["kernel"][idx], patch11[idx] * res["kernel"][idx],
        patch12[idx], res["kernel"][idx], patch12[idx] * res["kernel"][idx],
        patch13[idx], res["kernel"][idx], patch13[idx] * res["kernel"][idx],
    ],
    ncols=3,
    viztype="global",
)

## Is kernel c[0] a blur for vertical edges?

In [ ]:
plt.imshow(cds.filled_circle(50))

In [ ]:
res["kernel"][0]

In [ ]:
# stripes = cds.stripes(50, 0, 18)
stripes = cds.stripes(50, 90, 12)
ellipse = cds.filled_ellipse(50, "vertical", 5.0)
step_edge = cds.step_edge(50, 90)
line = cds.single_line(50, 90)
black = cds.black(50)
white = cds.white(50)
impulse = cds.unit_impulse(50)

ip_tensor = (stack_batch([impulse, black, black]))

res = run_and_show_for_input(layer_id, ip_tensor, learner.model, 0, viztype="local")

In [ ]:
idx = 0
show_single_channel_red_green_black(
    [
        res["input"][idx], res["per_chan"][idx], res["kernel"][idx],
        # patch11[idx], res["kernel"][idx], patch11[idx] * res["kernel"][idx],
        # patch12[idx], res["kernel"][idx], patch12[idx] * res["kernel"][idx],
        # patch13[idx], res["kernel"][idx], patch13[idx] * res["kernel"][idx],
    ],
    (15,15),
    ncols=3,
    viztype="global",
)

In [ ]:
chan0 = res["per_chan"][0]
chan0[10][9], chan0[10][10], chan0[10][11], chan0[10][12], chan0[10][13], chan0[10][14]
# ip = stack_batch(res["input"])
# patch11, bbox11 = get_input_patch_padded(ip, layer_id, learner.model, (10, 11))
# patch11 = patch11.numpy()[0]
# patch12, bbox12 = get_input_patch_padded(ip, layer_id, learner.model, (10, 12))
# patch12 = patch12.numpy()[0]
# patch13, bbox13 = get_input_patch_padded(ip, layer_id, learner.model, (10, 13))
# patch13 = patch13.numpy()[0]

# print("patch shape", patch11.shape, patch12.shape, patch13.shape)
# print("10,11")
# print_green_and_red_stat(res["per_chan"], 10, 11)
# print("10,12")
# print_green_and_red_stat(res["per_chan"], 10, 12)
# print("10,13")
# print_green_and_red_stat(res["per_chan"], 10, 13)

In [ ]:
import cv2
import numpy as np

# These parameters approximate your 7x7 array
gabor_kernel = cv2.getGaborKernel(ksize=(7, 7), 
                            sigma=1.6, 
                            theta=np.pi/2, 
                            lambd=2.8, 
                            gamma=0.5, 
                            psi=np.pi/2)

In [ ]:
show_single_channel_red_green_black([gabor_kernel, res["kernel"][0]])

In [ ]:
lgrad = cds.linear_gradient(50, 0)
ip_tensor = stack_batch([lgrad, black, black])

res = run_and_show_for_input(layer_id, ip_tensor, learner.model, 0, viztype="global")